In [96]:

"""
This script provides the computer-assisted sign checks used in Appendix A, Step 3
(of the paper “Higher-order estimates of a highest wave of the Whitham equation”).

In Appendix A, Step 3 we introduce f(σ) and split the proof into the three σ-ranges
(1,2], [2,4], and [4,∞). In each range, the paper derives an explicit lower bound
for f(σ) that depends only on s and fixed constants (a=0.72, c=0.5). The paper then
partitions s ∈ [1/2, 1) into subintervals [s_i, s_{i+1}] of length step = 10^{-3}
and uses monotonicity in s (proved analytically in the text) to reduce each bound
to an endpoint expression on each [s_i, s_{i+1}].

The present code implements exactly those endpoint expressions and evaluates them
using interval arithmetic (mpmath.iv) with mp.dps = 200. Each computed value is a
certified enclosure [L_i, U_i] of the corresponding bound on the interval [s_i, s_{i+1}].
The script records and prints the “worst margin”, i.e. min_i L_i, for each of the
three checks. If the worst margin is strictly positive, then the required inequality
holds for every i in the partition, hence for all s ∈ [1/2, 1).

Implementation notes:
  • Beta is evaluated via Γ(s)^2 / Γ(2s).
  • atanh(c) is computed via 0.5*log((1+c)/(1−c)).
  • The output “worst margin” is the smallest certified lower endpoint across i.
"""


'\nThis script provides the computer-assisted sign checks used in Appendix A, Step 3\n(of the paper “Higher-order estimates of a highest wave of the Whitham equation”).\n\nIn Appendix A, Step 3 we introduce f(σ) and split the proof into the three σ-ranges\n(1,2], [2,4], and [4,∞). In each range, the paper derives an explicit lower bound\nfor f(σ) that depends only on s and fixed constants (a=0.72, c=0.5). The paper then\npartitions s ∈ [1/2, 1) into subintervals [s_i, s_{i+1}] of length step = 10^{-3}\nand uses monotonicity in s (proved analytically in the text) to reduce each bound\nto an endpoint expression on each [s_i, s_{i+1}].\n\nThe present code implements exactly those endpoint expressions and evaluates them\nusing interval arithmetic (mpmath.iv) with mp.dps = 200. Each computed value is a\ncertified enclosure [L_i, U_i] of the corresponding bound on the interval [s_i, s_{i+1}].\nThe script records and prints the “worst margin”, i.e. min_i L_i, for each of the\nthree checks. If

In [97]:
from mpmath import mp, iv
import numpy as np

In [ ]:
mp.dps = 200
a = iv.mpf('0.72')
c = iv.mpf('0.5')

In [89]:
def iv_atanh(x):
    return iv.mpf('0.5') * iv.log((iv.mpf('1') + x) / (iv.mpf('1') - x))

def iv_pow(base,exp):
    return iv.mpf(str(base))**iv.mpf(str(exp))

def beta_iv(s):
    return iv.gamma(s)*iv.gamma(s)/iv.gamma(2*s)

def b_s_upper(s, a=0.72):
    a      = iv.mpf(a)
    s    = iv.mpf(s)      # example

    b_s_upper = (
            (iv_pow(a, iv.mpf('2') * s) / s)
            - (iv.mpf('2') * iv_pow(a, s + iv.mpf('1')) / (s + iv.mpf('1')))
        )
        
    return b_s_upper

def b_s_lower_bound(s_i,s_ip1, c=0.5):
    c      = iv.mpf(c)
    s_i    = iv.mpf(s_i)      # example
    s_ip1  = iv.mpf(s_ip1)

    b_s_lower_bound = (
            (iv_pow(c, iv.mpf('2') * s_ip1) / s_ip1)
            - (iv.mpf('2') * iv_pow(c, s_i + iv.mpf('1')) / (s_i + iv.mpf('1')))
            - (iv_pow(c, s_i + iv.mpf('3')) / (s_i + iv.mpf('3')))
            - (iv.mpf('2') * iv_atanh(c))
            + ((iv.mpf('2') / iv.mpf('3')) * iv_pow(c, iv.mpf('3')))
            + (iv.mpf('2') * c)
        )
        
    return b_s_lower_bound

In [95]:
step=0.001
s_i_list = np.arange(0.5,1,step)

def check_1(s_i_list, step):
    # \sigma \in (1,2]
    # Check for B(s,s)-5b_s>0.
    
    interval_list = []
    worst = None
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        val = beta_iv(S_ip1)-iv.mpf(5)*b_s_upper(S_i)
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst


def check_2(s_i_list, step,a=0.72):
    # \sigma \in [2,4]
    # Check for h^s_1(4)+h^s_2(4)+h^s_3(2)>0.
    
    interval_list = []
    worst = None
    
    A=iv.mpf(a)
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        val = (
            (iv.mpf(0.25)-iv.mpf(2)*iv_pow(A,S_i))/S_i
            -iv_pow(iv.mpf(4),(iv.mpf(0.5)-iv.mpf(1)/S_ip1))*S_ip1/(S_ip1+iv.mpf(1))
            +iv.mpf(2)*A
            +(iv.mpf(3)/iv.mpf(8))*beta_iv(S_ip1)
            +iv.mpf(0.75)*b_s_lower_bound(S_i,S_ip1)
        )
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst

def check_3(s_i_list, step,a=0.72):
    # \sigma \in [4, \infty)
    # Check for h^s_1(\infty)+h^s_2(\infty)+h^s_3(4)>0.
    
    interval_list = []
    worst = None
    
    A=iv.mpf(a)
    for s_i in s_i_list:
        S_i = iv.mpf(s_i)
        S_ip1 = iv.mpf(s_i+step)
        val = (
            (-iv.mpf(2)*iv_pow(A,S_i))/S_i
            +iv.mpf(2)*A
            +beta_iv(S_ip1)/iv.mpf(2)
            +(iv.mpf(13)/iv.mpf(16))*b_s_lower_bound(S_i,S_ip1)
        )
        interval_list.append(val)
        
        L = val.a
        if worst is None or L<worst:
            worst=L
        
    return interval_list, worst


In [91]:
check_1_result, check_1_smallest = check_1(s_i_list,step)
check_2_result, check_2_smallest = check_2(s_i_list,step)
check_3_result, check_3_smallest = check_3(s_i_list,step)

In [92]:
print(check_1_smallest)
print(check_2_smallest)
print(check_3_smallest)

[0.005834578701250592303, 0.005834578701250592303]
[0.042284149671170889739, 0.042284149671170889739]
[0.006143068318802402672, 0.006143068318802402672]


In [93]:
# All are >0, so the proof holds.